# ML-1M: SparseWalker + two causal temporal layers

Goal: test whether **depth**, not width, closes the remaining gap to SASRec.

Architecture:

`corrected Walker v1.1 -> temporal block #1 -> temporal block #2 -> tied scorer`

Each temporal block is exactly the same shape as the successful one-layer control: `d=64`, 2 heads, FFN x4, dropout 0.1. Training is from scratch with the same canonical ML-1M FullCE protocol.

At each validation checkpoint, the same trained model is evaluated at temporal depth **0 / 1 / 2**. The headline comparison is the independently trained depth-2 score versus the completed one-layer score (~0.161) and SASRec (~0.1864).


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, torch
REPO='/content/Sparsewalker'
BRANCH='agent/walker-two-temporal-layers'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
sys.path.insert(0,f'{REPO}/src')
sys.path.insert(0,f'{REPO}/experiments')
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'):
        del sys.modules[name]
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH)


## Run

Watch these fields:

- `val_depth2_NDCG@10`: full two-layer model
- `val_depth1_NDCG@10`: same trained weights with layer 2 bypassed
- `val_depth0_NDCG@10`: Walker-only path
- `layer2_contribution`: depth-2 minus depth-1
- `eval_cost_ratio_depth2_vs_depth0`: actual evaluation overhead

A causal-leak test runs before training.


In [ ]:
import runpy, sys
SCRIPT=f'{REPO}/experiments/run_ml1m_walker_two_temporal_layers.py'
sys.argv=[
    SCRIPT,
    '--seed','42',
    '--max-epochs','50',
    '--eval-every','5',
    '--patience','20',
    '--batch-size','128',
    '--eval-batch-size','1024',
    '--attn-heads','2',
    '--ff-mult','4',
    '--dropout','0.1',
]
print('INPROCESS WALKER+2TEMPORAL START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('INPROCESS WALKER+2TEMPORAL END',flush=True)


## Inspect

The first useful checkpoints are epochs 5, 10, and 15. If depth-2 is already clearly above the one-layer trajectory, keep it running.


In [ ]:
import json, pandas as pd
from pathlib import Path
root=Path('/content/drive/MyDrive/sparsewalker_two_temporal_layers/ml1m/seed42')
hp=root/'history.csv'
rp=root/'result.json'
if hp.exists(): display(pd.read_csv(hp))
if rp.exists(): print(json.dumps(json.loads(rp.read_text()),indent=2))
